# 第13章：高级技术补充

本章深入Agent开发中的高级技术，涵盖模型微调（Fine-tuning）、知识蒸馏（Distillation）、A/B实验框架和Prompt版本管理。这些技术帮助你将Agent从原型提升到生产级别。

**核心知识点**：
- Agent微调：通过Fine-tuning让模型适应特定领域
- 模型蒸馏：将大模型知识迁移到小模型
- A/B实验：科学评估Agent改进效果
- Prompt版本管理：系统化管理提示词迭代

**难度等级**：⭐⭐⭐⭐ 进阶


## 学习目标与环境准备

本章需要安装额外依赖：
```bash
pip install openai scipy matplotlib
```

**学习目标**：
1. 理解Agent微调的核心流程
2. 掌握模型蒸馏的基本方法
3. 学会构建A/B实验框架
4. 掌握Prompt版本管理策略


## 13.1 Agent微调（Fine-tuning）

微调通过在特定领域数据上继续训练模型，让Agent在垂直领域表现更好。下面实现一个微调数据准备和管理工具。


In [ ]:
import json
import os
from typing import List, Dict, Optional
from dataclasses import dataclass, field
from datetime import datetime


@dataclass
class FineTuningExample:
    system_prompt: str
    user_message: str
    assistant_response: str

    def to_openai_format(self) -> Dict:
        return {
            "messages": [
                {"role": "system", "content": self.system_prompt},
                {"role": "user", "content": self.user_message},
                {"role": "assistant", "content": self.assistant_response}
            ]
        }


@dataclass
class FineTuningDataset:
    name: str
    examples: List[FineTuningExample] = field(default_factory=list)
    version: str = "1.0.0"
    created_at: str = field(default_factory=lambda: datetime.now().isoformat())

    def add_example(self, system: str, user: str, assistant: str):
        self.examples.append(FineTuningExample(system, user, assistant))

    def to_jsonl(self, filepath: str):
        with open(filepath, "w", encoding="utf-8") as f:
            for ex in self.examples:
                line = json.dumps(ex.to_openai_format(), ensure_ascii=False)
                f.write(line + "
")
        return filepath

    def split(self, train_ratio: float = 0.8):
        split_idx = int(len(self.examples) * train_ratio)
        train = FineTuningDataset(f"{self.name}_train", self.examples[:split_idx])
        val = FineTuningDataset(f"{self.name}_val", self.examples[split_idx:])
        return train, val


dataset = FineTuningDataset("customer_service_agent")
dataset.add_example(
    "你是一个专业的客服Agent，回答简洁专业。",
    "我的订单什么时候发货？",
    "您的订单将在下单后24小时内发货，您可以在订单详情页查看物流信息。"
)
dataset.add_example(
    "你是一个专业的客服Agent，回答简洁专业。",
    "如何申请退款？",
    "您可以在订单页面点击申请退款按钮，我们将在3-5个工作日内处理您的退款申请。"
)
dataset.add_example(
    "你是一个专业的客服Agent，回答简洁专业。",
    "产品有问题怎么处理？",
    "非常抱歉给您带来不便。请提供订单号和问题照片，我们会在48小时内为您处理换货或退款。"
)

print(f"数据集: {dataset.name}")
print(f"示例数: {len(dataset.examples)}")

train, val = dataset.split(train_ratio=0.8)
print(f"训练集: {len(train.examples)} 条")
print(f"验证集: {len(val.examples)} 条")

print("
--- 第一条训练数据 ---")
print(json.dumps(dataset.examples[0].to_openai_format(), ensure_ascii=False, indent=2))


## 13.2 模型蒸馏（Knowledge Distillation）

蒸馏将大模型（教师）的知识迁移到小模型（学生），在保持性能的同时降低推理成本。


In [ ]:
import numpy as np
from typing import List, Tuple
from dataclasses import dataclass


@dataclass
class DistillationConfig:
    temperature: float = 3.0
    alpha: float = 0.5


class KnowledgeDistiller:
    def __init__(self, config: DistillationConfig = None):
        self.config = config or DistillationConfig()
        self.history: List[Dict] = []

    def softmax_with_temperature(self, logits: np.ndarray, temp: float) -> np.ndarray:
        exp_logits = np.exp(logits / temp)
        return exp_logits / exp_logits.sum(axis=-1, keepdims=True)

    def distillation_loss(
        self,
        student_logits: np.ndarray,
        teacher_logits: np.ndarray,
        hard_labels: np.ndarray
    ) -> Tuple[float, float, float]:
        T = self.config.temperature

        teacher_soft = self.softmax_with_temperature(teacher_logits, T)
        student_soft = self.softmax_with_temperature(student_logits, T)

        soft_loss = -np.sum(teacher_soft * np.log(student_soft + 1e-10))

        student_hard = self.softmax_with_temperature(student_logits, 1.0)
        hard_loss = -np.sum(hard_labels * np.log(student_hard + 1e-10))

        alpha = self.config.alpha
        total_loss = alpha * soft_loss + (1 - alpha) * hard_loss

        return total_loss, soft_loss, hard_loss

    def simulate_distillation(self) -> Dict:
        config = self.config

        teacher_logits = np.array([2.5, 0.8, 0.3, 0.1])
        hard_labels = np.array([1.0, 0.0, 0.0, 0.0])

        noise = np.random.normal(0, 0.3, size=teacher_logits.shape)
        student_logits = teacher_logits + noise

        total, soft, hard = self.distillation_loss(
            student_logits, teacher_logits, hard_labels
        )

        teacher_probs = self.softmax_with_temperature(teacher_logits, 1.0)
        student_probs = self.softmax_with_temperature(student_logits, 1.0)

        return {
            "teacher_probs": teacher_probs.tolist(),
            "student_probs": student_probs.tolist(),
            "soft_loss": round(soft, 4),
            "hard_loss": round(hard, 4),
            "total_loss": round(total, 4),
            "temperature": config.temperature,
            "alpha": config.alpha
        }


distiller = KnowledgeDistiller()
result = distiller.simulate_distillation()

print(f"温度参数 T: {result['temperature']}")
print(f"平衡系数 alpha: {result['alpha']}")
print(f"
教师模型概率分布: {[round(p, 4) for p in result['teacher_probs']]}")
print(f"学生模型概率分布: {[round(p, 4) for p in result['student_probs']]}")
print(f"
软损失（Soft Loss）: {result['soft_loss']}")
print(f"硬损失（Hard Loss）: {result['hard_loss']}")
print(f"总损失（Total Loss）: {result['total_loss']}")


## 13.3 A/B实验框架

A/B实验是科学评估Agent改进效果的核心方法。下面构建一个支持流量分配、统计检验和可视化的A/B实验框架。


In [ ]:
import random
import math
from typing import Dict, List, Any, Callable
from dataclasses import dataclass, field
from enum import Enum


class Variant(Enum):
    CONTROL = "control"
    TREATMENT = "treatment"


@dataclass
class ABExperiment:
    name: str
    control_fn: Callable
    treatment_fn: Callable
    traffic_split: float = 0.5
    
    control_results: List[float] = field(default_factory=list)
    treatment_results: List[float] = field(default_factory=list)

    def run(self, input_data: Any) -> Dict:
        variant = Variant.TREATMENT if random.random() < self.traffic_split else Variant.CONTROL

        if variant == Variant.CONTROL:
            result = self.control_fn(input_data)
            self.control_results.append(result.get("score", 0))
        else:
            result = self.treatment_fn(input_data)
            self.treatment_results.append(result.get("score", 0))

        return {"variant": variant.value, **result}

    def analyze(self) -> Dict:
        if len(self.control_results) < 2 or len(self.treatment_results) < 2:
            return {"error": "数据不足，至少需要2个样本"}

        c_mean = sum(self.control_results) / len(self.control_results)
        t_mean = sum(self.treatment_results) / len(self.treatment_results)
        
        c_var = sum((x - c_mean) ** 2 for x in self.control_results) / (len(self.control_results) - 1)
        t_var = sum((x - t_mean) ** 2 for x in self.treatment_results) / (len(self.treatment_results) - 1)

        se = math.sqrt(c_var / len(self.control_results) + t_var / len(self.treatment_results))
        t_stat = (t_mean - c_mean) / se if se > 0 else 0
        lift = ((t_mean - c_mean) / c_mean * 100) if c_mean > 0 else 0

        return {
            "control_samples": len(self.control_results),
            "treatment_samples": len(self.treatment_results),
            "control_mean": round(c_mean, 4),
            "treatment_mean": round(t_mean, 4),
            "lift_percent": round(lift, 2),
            "t_statistic": round(t_stat, 4),
            "significant": abs(t_stat) > 1.96
        }


def control_agent(query: str) -> Dict:
    base_score = 0.75 + random.gauss(0, 0.08)
    return {"response": f"[Control] 收到问题: {query}", "score": base_score}

def treatment_agent(query: str) -> Dict:
    improved_score = 0.82 + random.gauss(0, 0.08)
    return {"response": f"[Treatment] 收到问题: {query}", "score": improved_score}


experiment = ABExperiment(
    name="客服响应质量测试",
    control_fn=control_agent,
    treatment_fn=treatment_agent,
    traffic_split=0.5
)

queries = [
    "如何重置密码？", "退货流程是什么？",
    "配送需要几天？", "如何联系客服？",
    "支持哪些支付方式？", "订单状态怎么查？",
    "优惠券如何使用？", "如何修改收货地址？",
    "发票怎么开具？", "商品缺货怎么办？"
]

for query in queries:
    result = experiment.run(query)

analysis = experiment.analyze()
print(f"实验: {experiment.name}")
print(f"对照组样本: {analysis['control_samples']}")
print(f"实验组样本: {analysis['treatment_samples']}")
print(f"对照组均分: {analysis['control_mean']}")
print(f"实验组均分: {analysis['treatment_mean']}")
print(f"提升幅度: {analysis['lift_percent']}%")
print(f"t统计量: {analysis['t_statistic']}")
print(f"统计显著: {'是' if analysis['significant'] else '否'}")


## 13.4 Prompt版本管理

系统化管理Prompt的迭代，支持语义化版本、分支管理和Canary发布。


In [ ]:
from dataclasses import dataclass, field
from typing import Dict, List, Optional
from datetime import datetime
import hashlib


@dataclass
class PromptVersion:
    version: str
    content: str
    description: str
    created_at: str
    parent_version: Optional[str] = None
    metadata: Dict = field(default_factory=dict)

    def hash(self) -> str:
        return hashlib.md5(self.content.encode()).hexdigest()[:8]


class PromptVersionManager:
    def __init__(self):
        self.versions: Dict[str, PromptVersion] = {}
        self.aliases: Dict[str, str] = {}
        self.canary_rules: Dict[str, Dict] = {}

    def create_version(self, version: str, content: str, description: str = "",
                       parent: str = None, metadata: Dict = None) -> PromptVersion:
        pv = PromptVersion(
            version=version,
            content=content,
            description=description,
            created_at=datetime.now().isoformat(),
            parent_version=parent,
            metadata=metadata or {}
        )
        self.versions[version] = pv
        return pv

    def set_alias(self, alias: str, version: str):
        if version not in self.versions:
            raise ValueError(f"版本 {version} 不存在")
        self.aliases[alias] = version

    def resolve(self, ref: str) -> PromptVersion:
        if ref in self.aliases:
            ref = self.aliases[ref]
        if ref not in self.versions:
            raise ValueError(f"版本或别名 '{ref}' 不存在")
        return self.versions[ref]

    def canary_deploy(self, old_version: str, new_version: str, rollout_pct: float):
        self.canary_rules[new_version] = {
            "previous": old_version,
            "rollout": rollout_pct,
            "started_at": datetime.now().isoformat()
        }

    def get_version_for_request(self, request_id: str = None) -> str:
        for new_ver, rule in self.canary_rules.items():
            import random
            if random.random() < rule["rollout"]:
                return new_ver
            return rule["previous"]
        return self.aliases.get("latest", list(self.versions.keys())[-1] if self.versions else "0.0.0")

    def history(self) -> List[Dict]:
        return [
            {
                "version": v.version,
                "description": v.description,
                "hash": v.hash(),
                "created_at": v.created_at,
                "parent": v.parent_version
            }
            for v in sorted(self.versions.values(), key=lambda x: x.created_at)
        ]


manager = PromptVersionManager()

v1 = manager.create_version("1.0.0",
    "你是客服Agent，回答简短。问题：{query}",
    "初始版本-基础客服")

v2 = manager.create_version("1.1.0",
    "你是一个专业的客服Agent。用友好、专业的语气回答用户问题。
问题：{query}",
    "优化语气，增加友好性",
    parent="1.0.0")

v3 = manager.create_version("2.0.0",
    "你是高级客服Agent，职责是解决用户问题。
首先理解用户意图，然后给出准确、有帮助的回复。
用户问题：{query}",
    "重大重构-增加意图理解和帮助性指导")

manager.set_alias("latest", "2.0.0")
manager.set_alias("stable", "1.1.0")

manager.canary_deploy("1.1.0", "2.0.0", rollout_pct=0.2)

print("=== Prompt版本历史 ===")
for entry in manager.history():
    print(f"v{entry['version']} ({entry['hash']}): {entry['description']}")
    if entry['parent']:
        print(f"  parent: v{entry['parent']}")

print(f"
别名:")
for alias, ver in manager.aliases.items():
    print(f"  {alias} -> {ver}")

print(f"
Canary部署:")
for ver, rule in manager.canary_rules.items():
    print(f"  v{ver}: rollout={rule['rollout']*100}% (previous=v{rule['previous']})")

print(f"
请求路由模拟:")
for i in range(5):
    assigned = manager.get_version_for_request(f"req_{i}")
    print(f"  req_{i} -> v{assigned}")


## 练习与思考

1. **微调数据准备**：为你的Agent场景准备10条微调数据，使用FineTuningDataset导出为JSONL格式。

2. **蒸馏实验**：修改温度参数T（1.0~5.0），观察软损失和硬损失的变化趋势。

3. **A/B测试**：增加实验样本量到100，观察统计显著性如何随样本量增加而变化。

4. **版本管理**：为你的Prompt系统添加分支管理功能，支持从任意版本创建分支。
